Conexão do Colab com o Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Visualização dos datasets

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/raw")

csv_files = list(RAW_DIR.rglob("*.csv"))

for file in csv_files:
    print(file)

Visualização das estruturas dos datasets

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/codefeedback/codefeedback_train.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/stackoverflow_questions/Answers.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/stackoverflow_questions/Questions.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/stackoverflow_questions/Tags.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/faq/faq_dataset.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/glaive_python_qa/train.csv",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/dataset_python_qa/Dataset_Python_Question_Answer.csv"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_csv(file, nrows=5)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")
    print(df.head())

Visualização das estruturas sem o erro utf-8

In [ ]:
import pandas as pd

for file in files:

    print("="*100)
    print(file)

    try:
        df = pd.read_csv(file, nrows=5)

    except UnicodeDecodeError:

        try:
            df = pd.read_csv(file, encoding="latin-1", nrows=5)

        except:

            df = pd.read_csv(file, encoding="cp1252", nrows=5)

    print(df.columns.tolist())
    print(df.head())

Criação da subpasta onde ficarão os datasets filtrados

In [ ]:
from pathlib import Path

RAW = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/raw")
FILTERED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/filtered")

FILTERED.mkdir(parents=True, exist_ok=True)

Filtragem dataset Codefeedback

In [ ]:
import pandas as pd

df = pd.read_csv(
    RAW/"codefeedback"/"codefeedback_train.csv"
)

df = df[df["lang"].str.lower() == "python"]

df = df.rename(columns={
    "query":"question"
})

df = df[[
    "question",
    "answer"
]]

df["source"] = "codefeedback"

df.to_parquet(
    FILTERED/"codefeedback_python.parquet",
    index=False
)

Dataset python convertido para parquet

In [ ]:
faq = pd.read_csv(
    RAW/"faq"/"faq_dataset.csv",
    encoding="latin-1"
)

faq = faq.rename(columns={
    "Question":"question",
    "Answer":"answer"
})

faq["source"] = "faq"

faq.to_parquet(
    FILTERED/"faq_python.parquet",
    index=False
)

Dataset python convertido para parquet

In [ ]:
glaive = pd.read_csv(
    RAW/"glaive_python_qa"/"train.csv"
)

glaive["source"] = "glaive"

glaive.to_parquet(
    FILTERED/"glaive_python.parquet",
    index=False
)

Filtragem dataset Stackoverflow

In [ ]:
import pandas as pd

questions = pd.read_csv(
    RAW / "stackoverflow_questions" / "Questions.csv",
    encoding="utf-8",
    encoding_errors="replace",
    low_memory=False
)

answers = pd.read_csv(
    RAW / "stackoverflow_questions" / "Answers.csv",
    encoding="utf-8",
    encoding_errors="replace",
    low_memory=False
)

tags = pd.read_csv(
    RAW / "stackoverflow_questions" / "Tags.csv",
    encoding="utf-8",
    encoding_errors="replace",
    low_memory=False
)

python_ids = (
    tags.loc[
        tags["Tag"].str.strip().str.lower() == "python",
        "Id"
    ].unique()
)

questions = questions[
    questions["Id"].isin(python_ids)
][[
    "Id",
    "Title",
    "Body",
    "Score"
]]

answers = answers[
    answers["ParentId"].isin(python_ids)
][[
    "ParentId",
    "Body",
    "Score"
]]

merged = answers.merge(
    questions,
    left_on="ParentId",
    right_on="Id",
    suffixes=("_answer", "_question")
)

merged["question"] = (
    merged["Title"].fillna("").str.strip()
    + "\n\n"
    + merged["Body_question"].fillna("").str.strip()
)

merged["answer"] = (
    merged["Body_answer"]
    .fillna("")
    .str.strip()
)

merged["question_id"] = merged["ParentId"]
merged["question_score"] = merged["Score_question"]
merged["answer_score"] = merged["Score_answer"]
merged["source"] = "stackoverflow"

merged = merged.dropna(subset=["question", "answer"])

merged = merged[
    (merged["question"] != "")
    & (merged["answer"] != "")
]


merged = merged.drop_duplicates(
    subset=["question", "answer"]
)

merged = merged[
    [
        "question_id",
        "question",
        "answer",
        "question_score",
        "answer_score",
        "source"
    ]
]

print("Quantidade de perguntas Python:", len(questions))
print("Quantidade de respostas Python:", len(answers))
print("Quantidade de pares pergunta-resposta:", len(merged))

print("\nPrimeiros registros:")
print(merged.head())

print("\nAmostra aleatória:")
print(
    merged.sample(5, random_state=42)[
        ["question", "answer"]
    ]
)

if merged.empty:
    raise ValueError(
        "Nenhum registro de Python foi encontrado."
    )

merged.to_parquet(
    FILTERED / "stackoverflow_python.parquet",
    index=False
)

print(f"\nRegistros finais: {len(merged):,}")
print(f"Arquivo salvo em: {FILTERED/'stackoverflow_python.parquet'}")
print("Dataset salvo com sucesso!")

Verificação da codificação do dataset


In [ ]:
import chardet

arquivo = RAW / "stackoverflow_questions" / "Questions.csv"

with open(arquivo, "rb") as f:
    resultado = chardet.detect(f.read(100000))

print(resultado)

Visualização das questões do dataset da StackOverflow

In [ ]:
questions = pd.read_csv(
    RAW / "stackoverflow_questions" / "Questions.csv",
    nrows=5
)

questions.head()

Verificando se a filtragem do dataset da Stackoverflow foi bem feita

In [ ]:
import pandas as pd
import re

df = pd.read_parquet("/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/stackoverflow_python.parquet")

codigo_mask = df["answer"].str.contains(r"```", na=False)
df_codigo = df[codigo_mask]

texto = (df_codigo["question"].fillna("") + " " + df_codigo["answer"].fillna("")).str.lower()

padroes = {
    "java": [
        r"public\s+static\s+void",
        r"system\.out\.println",
        r"import\s+java"
    ],
    "c#": [
        r"console\.writeline",
        r"using\s+system",
        r"namespace\s+\w+"
    ],
    "php": [
        r"<\?php",
        r"\$\w+\s*=",
        r"echo\s+\$"
    ],
    "cpp": [
        r"#include\s*<",
        r"std::cout",
        r"using\s+namespace\s+std"
    ],
    "javascript": [
        r"console\.log\(",
        r"function\s+\w+\s*\(",
        r"=>"
    ],
    "ruby": [
        r"\bdef\s+\w+",
        r"\bend\b"
    ]
}

for lang, patterns in padroes.items():
    mask = texto.apply(lambda x: any(re.search(p, x, re.IGNORECASE) for p in patterns))
    print(f"{lang}: {mask.sum()} possíveis ocorrências")

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/raw")
FILTERED = Path("/content/drive/MyDrive/TCC_Chatbot/datasets/filtered")

FILTERED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    RAW / "dataset_python_qa" / "Dataset_Python_Question_Answer.csv"
)

df = df.rename(columns={
    "Question": "question",
    "Answer": "answer"
})

df["source"] = "dataset_python_qa"

df.to_parquet(
    FILTERED / "dataset_python_qa.parquet",
    index=False
)

print("Dataset salvo com sucesso!")
print(df.head())

Remoção de duplicatas do dataset

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/raw/code_qa_updated/code_qa_updated.parquet"
)

print("Registros originais:", len(df))

df = df[["question", "code", "answer"]]

df = df.dropna(subset=["question", "code", "answer"])

for col in ["question", "code", "answer"]:
    df[col] = df[col].str.strip()

df = df[
    (df["question"] != "") &
    (df["code"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates()

print("Registros após limpeza:", len(df))

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/code_qa_updated.parquet",
    index=False
)

print("Dataset salvo!")

display(df.head())